# 01 — Data acquisition and quality audit

## Objective

Download the Home Credit Default Risk data securely, keep credentials outside version control, and create a reproducible baseline data-quality audit.

## Inputs and outputs

- **Input:** Kaggle API credentials stored locally in `.env`.
- **Output:** source CSV files in `data/Raw/` and `reports/data_quality_audit.csv`.

> Do not commit Kaggle credentials or raw data. This notebook is the canonical entry point for data acquisition; `01_data_collection.ipynb` is retained only as a legacy reference.


In [ ]:
import os
import zipfile
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "Raw"
PROCESSED_DATA_DIR = DATA_DIR / "Processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("KAGGLE_USERNAME") or not os.getenv("KAGGLE_KEY"):
    raise EnvironmentError(
        "KAGGLE_USERNAME and KAGGLE_KEY are required. "
        "Create a local .env file before running this notebook."
    )

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()
api.competition_download_files("home-credit-default-risk", path=RAW_DATA_DIR)

archive_path = RAW_DATA_DIR / "home-credit-default-risk.zip"
if archive_path.exists():
    with zipfile.ZipFile(archive_path, "r") as archive:
        archive.extractall(RAW_DATA_DIR)
    archive_path.unlink()

print(f"Data available in: {RAW_DATA_DIR}")


## Quality audit

The audit records dimensions and missingness by file. It is intended as an early warning for absent files, schema changes, or unusually sparse data before downstream modelling begins.


In [ ]:
audit_records = []

for csv_path in sorted(RAW_DATA_DIR.glob("*.csv")):
    frame = pd.read_csv(csv_path)
    total_cells = frame.shape[0] * frame.shape[1]
    missing_values = int(frame.isna().sum().sum())

    audit_records.append(
        {
            "file": csv_path.name,
            "rows": frame.shape[0],
            "columns": frame.shape[1],
            "missing_values": missing_values,
            "missing_pct": round(100 * missing_values / total_cells, 2) if total_cells else 0,
        }
    )

audit = pd.DataFrame(audit_records).sort_values("file").reset_index(drop=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
audit.to_csv(REPORTS_DIR / "data_quality_audit.csv", index=False)
audit
